In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ==========================================================
# Configuration
# ==========================================================

BRONZE_TABLE = "finance_intelligence_hub.bronze.company_news_polygon"
SILVER_TABLE = "finance_intelligence_hub.silver.company_news_polygon"

print("="*80)
print("Company News Silver Pipeline")
print("="*80)

# ==========================================================
# Helper Functions
# ==========================================================

def blank_to_null(column):
    c = F.trim(column)
    return F.when(c == "", F.lit(None)).otherwise(c)


def clean_text(column):
    c = blank_to_null(column)
    c = F.regexp_replace(c, r"[\r\n\t]+", " ")
    c = F.regexp_replace(c, r"\s{2,}", " ")
    return F.trim(c)


def parse_related_tickers(column):
    c = blank_to_null(column)
    arr = F.split(F.coalesce(c, F.lit("")), ",")
    arr = F.transform(arr, lambda x: F.upper(F.trim(x)))
    arr = F.filter(arr, lambda x: x != "")
    arr = F.array_distinct(arr)
    return arr


def is_valid_url(column):
    c = blank_to_null(column)
    return c.rlike(r"^https?://")


def extract_author_email(column):
    c = blank_to_null(column)
    email = F.regexp_extract(c, r"([A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,})", 1)
    return F.when(email != "", email).otherwise(F.lit(None))


def extract_author_name(column):
    c = blank_to_null(column)
    without_email = F.trim(F.regexp_replace(c, r"^\S+@\S+", ""))
    name_in_parens = F.regexp_extract(without_email, r"\(([^)]+)\)", 1)
    return F.when(name_in_parens != "", name_in_parens).otherwise(c)


def parse_dwh_loaded_at(column):
    """
    Bronze has historically stored dwh_loaded_at in two different string
    formats ('2026-07-04T00:00:00Z' from the backfill, and
    '2026-07-04 00:12:33' from the live incremental script). try_to_timestamp
    returns NULL on a format mismatch instead of raising (unlike
    to_timestamp under ANSI mode), so trying both formats and coalescing
    is safe.
    """
    c = blank_to_null(column)
    iso_fmt = F.try_to_timestamp(c, F.lit("yyyy-MM-dd'T'HH:mm:ss'Z'"))
    plain_fmt = F.try_to_timestamp(c, F.lit("yyyy-MM-dd HH:mm:ss"))
    return F.coalesce(iso_fmt, plain_fmt)


# ==========================================================
# Create Silver Table
# ==========================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE}
(
article_id STRING,
ticker STRING,
related_tickers ARRAY<STRING>,
related_tickers_count INT,
title STRING,
description STRING,
text_for_finbert STRING,
text_word_count INT,
author STRING,
author_name STRING,
author_email STRING,
publisher_name STRING,
publisher_url STRING,
source_url STRING,
image_url STRING,
image_url_is_valid BOOLEAN,
published_at TIMESTAMP,
published_date_raw STRING,
dwh_loaded_at TIMESTAMP,
silver_loaded_at TIMESTAMP
)
USING DELTA
""")

print("Silver table verified successfully.")

# ==========================================================
# Detect whether bronze has an ingestion-audit column
# ==========================================================

bronze_columns = spark.table(BRONZE_TABLE).columns
HAS_AUDIT_COL = "dwh_loaded_at" in bronze_columns

if HAS_AUDIT_COL:
    print("dwh_loaded_at column found -> running incremental load.")
else:
    print("WARNING: dwh_loaded_at column NOT found in bronze table.")
    print("Falling back to FULL LOAD every run (dedup + MERGE still keep it idempotent).")

# ==========================================================
# Incremental / Full Load
# ==========================================================

if HAS_AUDIT_COL:

    last_loaded = spark.sql(
        f"SELECT COALESCE(MAX(dwh_loaded_at), TIMESTAMP('1900-01-01')) FROM {SILVER_TABLE}"
    ).first()[0]

    bronze_df = (
        spark.table(BRONZE_TABLE)
        # cast explicitly BEFORE filtering, so the watermark comparison
        # is timestamp-vs-timestamp, not string-vs-timestamp
        .withColumn("dwh_loaded_at", parse_dwh_loaded_at(F.col("dwh_loaded_at")))
        .filter(F.col("dwh_loaded_at") > last_loaded)
    )

else:

    bronze_df = spark.table(BRONZE_TABLE)
    bronze_df = bronze_df.withColumn("dwh_loaded_at", F.lit(None).cast("timestamp"))

rows_read = bronze_df.count()
print(f"Rows Read : {rows_read}")

if rows_read == 0:
    dbutils.notebook.exit("No new records.")

# ==========================================================
# Data Cleaning
# النوع ده من الجداول مفيهوش business rules زي الأرقام
# المطلوب هنا "تنضيف" مش "رفض"، فمفيش quarantine table ومفيش
# صفوف بتتشال - بالظبط زي ما اتفقنا.
# ==========================================================

clean_df = (

    bronze_df

    #########################################################
    # Ticker / Related Tickers
    #########################################################
    .withColumn("ticker", F.upper(blank_to_null(F.col("ticker"))))
    .withColumn("related_tickers", parse_related_tickers(F.col("related_tickers")))
    .withColumn("related_tickers_count", F.size(F.col("related_tickers")))

    #########################################################
    # Text Fields (title / description / text_for_finbert)
    #########################################################
    .withColumn("title", clean_text(F.col("title")))
    .withColumn("description", clean_text(F.col("description")))
    .withColumn("text_for_finbert", clean_text(F.col("text_for_finbert")))
    .withColumn(
        "text_word_count",
        F.when(
            F.col("text_for_finbert").isNotNull(),
            F.size(F.split(F.col("text_for_finbert"), r"\s+"))
        ).otherwise(F.lit(0))
    )

    #########################################################
    # Author
    #########################################################
    .withColumn("author", blank_to_null(F.col("author")))
    .withColumn("author_name", extract_author_name(F.col("author")))
    .withColumn("author_email", extract_author_email(F.col("author")))

    #########################################################
    # Publisher / URLs
    #########################################################
    .withColumn("publisher_name", blank_to_null(F.col("publisher_name")))
    .withColumn("publisher_url", blank_to_null(F.col("publisher_url")))
    .withColumn("source_url", blank_to_null(F.col("source_url")))
    .withColumn("image_url", blank_to_null(F.col("image_url")))
    .withColumn("image_url_is_valid", is_valid_url(F.col("image_url")))

    #########################################################
    # Published Date
    # to_timestamp بترجع NULL تلقائيًا لو الفورمات غلط
    # (من غير ما توقف الكود زي cast العادي)
    #########################################################
    .withColumn("published_date_raw", F.col("published_date"))
    .withColumn(
        "published_at",
        F.try_to_timestamp(F.col("published_date"), F.lit("yyyy-MM-dd'T'HH:mm:ss'Z'"))
    )

    #########################################################
    # Surrogate Key
    # article_id ثابت لكل مقال، مبني على source_url
    # (أو على title + published_date لو مفيش source_url)
    #########################################################
    .withColumn(
        "article_id",
        F.sha2(
            F.coalesce(
                F.col("source_url"),
                F.concat_ws("||", F.col("title"), F.col("published_date_raw"))
            ),
            256
        )
    )

    # dwh_loaded_at is already TIMESTAMP by this point (cast happened on
    # bronze_df above) — no further transform needed here, it just carries
    # through with the correct type this time.

    #########################################################
    # Audit
    #########################################################
    .withColumn("silver_loaded_at", F.current_timestamp())

)

print(f"Rows after cleaning : {clean_df.count()}")

# ==========================================================
# Remove Exact Duplicates
# نفس المقال ممكن يتكرر (نفس source_url) لو الـ scraper
# سحبه أكتر من مرة - بناخد أحدث نسخة بس، من غير ما نرفض حاجة
# ==========================================================

window_spec = (
    Window
    .partitionBy("article_id")
    .orderBy(
        F.col("dwh_loaded_at").desc_nulls_last(),
        F.col("published_at").desc_nulls_last()
    )
)

silver_df = (
    clean_df
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

rows_valid = silver_df.count()
print(f"Rows After Dedup : {rows_valid}")



# ==========================================================
# Merge Into Silver
# ==========================================================

print("=" * 80)
print("Starting MERGE...")
print("=" * 80)

delta_table = DeltaTable.forName(spark, SILVER_TABLE)

(
    delta_table.alias("target")
    .merge(silver_df.alias("source"), "target.article_id = source.article_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE completed successfully.")

# ==========================================================
# Optimize Delta Table
# ==========================================================

print("Running OPTIMIZE...")
spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (ticker, published_at)")
print("Optimization completed.")

# ==========================================================
# Pipeline Metrics
# ==========================================================

total_rows = spark.sql(f"SELECT COUNT(*) FROM {SILVER_TABLE}").first()[0]
total_articles = spark.sql(f"SELECT COUNT(DISTINCT article_id) FROM {SILVER_TABLE}").first()[0]

print()
print("="*80)
print("Pipeline Summary")
print("="*80)
print(f"Rows Read            : {rows_read:,}")
print(f"Rows After Dedup     : {rows_valid:,}")
print(f"Distinct Articles    : {total_articles:,}")
print(f"Total Silver Rows    : {total_rows:,}")
print("="*80)